In [1]:
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader , Dataset,random_split
from torchvision import transforms, datasets
from torch.optim import Adam
from sklearn.metrics import confusion_matrix , ConfusionMatrixDisplay
#from torch.utilis.data import random_split
from torchvision.datasets import CIFAR10
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from PIL import Image
from torchvision.datasets import ImageFolder
import os
from torchsummary import summary

In [2]:
train_directory = "/content/drive/MyDrive/dataset/Training_maize_apple/corn"
test_directory = "/content/drive/MyDrive/dataset/Training_maize_apple/test"


In [3]:
def ConvertToRgb(image):
  if image.mode != 'RGB':
    return image.convert('RGB')
  else:
      return image
transform_p = transforms.Compose([
      transforms.Lambda(ConvertToRgb),
      transforms.Resize((224, 224)),
      transforms.RandomHorizontalFlip(p=1),
      transforms.RandomRotation(degrees=30),
      transforms.RandomVerticalFlip(p=1),
      transforms.ToTensor()
])


In [4]:
train_dataset = datasets.ImageFolder(root=train_directory, transform= transform_p)
test_dataset = datasets.ImageFolder(root =test_directory, transform= transform_p)

In [5]:
g= torch.Generator().manual_seed(42)
train_dataset , validation_dataset = random_split(train_dataset , [0.8,0.2],generator=g)
len(train_dataset),len(validation_dataset)


(36, 9)

In [6]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)
#test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [7]:
#creating a fuction that can take a path annd transform , split and get d train loader
def get_path (path=""):
  def ConvertToRgb(image):
    if image.mode != 'RGB':
      return image.convert('RGB')
    else:
      return image
  transform_p = transforms.Compose([
      transforms.Lambda(ConvertToRgb),
      transforms.Resize((360, 224)),
      transforms.RandomHorizontalFlip(p=1),
      transforms.RandomRotation(degrees=30),
      transforms.RandomVerticalFlip(p=1),
        transforms.ToTensor()
    ] )
  train_dataset = datasets.ImageFolder(root=path, transform= transform_p)
  val_dataset= datasets.ImageFolder(root=path, transform= transform_p)
  g= torch.Generator().manual_seed(42)
  train_dataset , validation_dataset = random_split(train_dataset , [0.8,0.2],generator=g)
  train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
  validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)
  return train_loader , validation_loader

In [8]:
train_loader , val_loader = get_path(train_directory)

In [9]:
next(iter(train_loader))[0].shape

torch.Size([32, 3, 360, 224])

In [10]:
# Define the model
model = torch.nn.Sequential()
model.add_module("conv1", nn.Conv2d(in_channels=3, out_channels=9, kernel_size=4, padding=1, stride=2))
model.add_module("relu1", nn.ReLU())
model.add_module("maxpool1", nn.MaxPool2d(kernel_size=2, stride=2))
model.add_module("conv2", nn.Conv2d(in_channels=9, out_channels=18, kernel_size=4, padding=1, stride=2))
model.add_module("relu2", nn.ReLU())
model.add_module("maxpool2", nn.MaxPool2d(kernel_size=2, stride=2))
model.add_module("conv3", nn.Conv2d(in_channels=18, out_channels=36, kernel_size=4, padding=1, stride=2))
model.add_module("relu3", nn.ReLU())
model.add_module("maxpool3", nn.MaxPool2d(kernel_size=2, stride=2))
model.add_module("flatten", nn.Flatten())
model.add_module("fc1", nn.Linear(in_features=540, out_features=100))  # Adjust in_features if needed
model.add_module("relu4", nn.ReLU())
model.add_module("fc2", nn.Linear(in_features=100, out_features=50))
model.add_module("relu5", nn.ReLU())
model.add_module("fc3", nn.Linear(in_features=50, out_features=3))
model.add_module("softmax", nn.Softmax(dim=1))  # Specify dim for Softmax

In [11]:
summary(model, (3, 360, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [-1, 9, 180, 112]             441
              ReLU-2          [-1, 9, 180, 112]               0
         MaxPool2d-3            [-1, 9, 90, 56]               0
            Conv2d-4           [-1, 18, 45, 28]           2,610
              ReLU-5           [-1, 18, 45, 28]               0
         MaxPool2d-6           [-1, 18, 22, 14]               0
            Conv2d-7            [-1, 36, 11, 7]          10,404
              ReLU-8            [-1, 36, 11, 7]               0
         MaxPool2d-9             [-1, 36, 5, 3]               0
          Flatten-10                  [-1, 540]               0
           Linear-11                  [-1, 100]          54,100
             ReLU-12                  [-1, 100]               0
           Linear-13                   [-1, 50]           5,050
             ReLU-14                   

In [12]:
model(next(iter(train_loader))[0]).shape

torch.Size([32, 3])

In [13]:
loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

In [15]:
from training import train,predict,train_epoch

In [16]:
train(model,optimizer,loss_fn,train_loader,val_loader,epochs=50)

Training:   0%|          | 0/2 [00:00<?, ?it/s]

Scoring:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 1, Training Loss: 1.10, Validation Loss: 1.10, Validation accuracy = 0.22


Training:   0%|          | 0/2 [00:00<?, ?it/s]

Scoring:   0%|          | 0/1 [00:00<?, ?it/s]

...

Scoring:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 49, Training Loss: 0.74, Validation Loss: 0.78, Validation accuracy = 0.89


Training:   0%|          | 0/2 [00:00<?, ?it/s]

Scoring:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch: 50, Training Loss: 0.75, Validation Loss: 0.80, Validation accuracy = 0.67


In [17]:
#creating a fuction that can take a path annd transform , split and get d train loader
def get_path (path=""):
  def ConvertToRgb(image):
    if image.mode != 'RGB':
      return image.convert('RGB')
    else:
      return image
  transform_p = transforms.Compose([
      transforms.Lambda(ConvertToRgb),
      transforms.Resize((360, 224)),
      transforms.RandomHorizontalFlip(p=1),
      transforms.RandomRotation(degrees=30),
      transforms.RandomVerticalFlip(p=1),
        transforms.ToTensor()
    ] )
  test_dataset = datasets.ImageFolder(root=path, transform= transform_p)
  #val_dataset= datasets.ImageFolder(root=path, transform= transform_p)
  g= torch.Generator().manual_seed(42)
  #train_dataset , validation_dataset = random_split(train_dataset , [0.8,0.2],generator=g)
  test_loader = DataLoader(test_dataset, batch_size=32)
  #validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)
  return test_loader

In [19]:
test_loader=get_path(test_directory)

In [22]:
for file in predictions:
  if file.argmax()==0:
    print("corn")
  elif file.argmax()==1:
    print("maize")
  elif file.argmax()==2:
    print("apple")

corn
apple
corn
corn
corn
corn
corn
corn
corn
apple
corn
corn
corn
corn
apple
maize
maize
maize
maize
maize
apple
apple
apple
apple
apple


In [ ]:
print(device)

In [ ]:
dataset= datasets.ImageFolder(root="/content/drive/MyDrive/dataset/test/Apple___Apple_scab/Apple___Apple_scab (1).JPG", transform= transform_p)

In [21]:
# in ipython-input-80-b20706dd7d80

# Assuming you want to run on GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predictions= predict(model, test_loader, device) # Pass device instead of loss_f
#predict(model,test_loader,loss_fn)

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]